# AdiVaani NMT — Part I: BERT Embeddings (LSTM Seq2Seq + Attention)
**Run this PARALLEL to Notebook 1. T4x2. Target: ~4-5 hours.**

Architecture:
- Hindi BERT encoder embeddings (`l3cube-pune/hindi-bert-v2`)
- Marathi BERT encoder embeddings (`l3cube-pune/marathi-bert-v2`)
- Frozen BERT + trainable projection → Bidirectional LSTM Encoder
- LSTM Decoder with Bahdanau Attention
- Mixed precision (AMP), DataParallel
- Supports frozen / unfrozen BERT comparison
- Outputs: loss curves, BLEU-100, CHRF++-100

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

pkgs = ['transformers', 'sentencepiece', 'sacrebleu', 'matplotlib', 'tqdm', 'accelerate']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])

print('All packages installed.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports
# ─────────────────────────────────────────────────────────────────────────────
import os, random, math, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel
import sentencepiece as spm
import sacrebleu
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Device: {DEVICE} | GPUs: {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Configuration
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # Data
    'data_dir': '/kaggle/input/datasets/au23cs060bodhinijain/adivani',
    'hi_file': 'train.hi',
    'mr_file': 'train.mr',
    'max_samples': 200_000,             # BERT encoding is slower; use fewer samples
    'max_len_words': 50,                # word-level filter before BERT tokenization
    'max_bert_len': 64,                 # max BERT tokens (src side)
    'val_split': 0.05,

    # BERT models
    'hi_bert': 'l3cube-pune/hindi-bert-v2',
    'mr_bert': 'l3cube-pune/marathi-bert-v2',
    'bert_freeze': True,                # True = frozen BERT (faster, less mem)
                                        # Set False to fine-tune BERT (slower)
    'bert_layers': 4,                   # number of last BERT layers to average

    # SentencePiece for TARGET side decoding
    'vocab_size': 8000,
    'spm_model': '/kaggle/working/spm_bert',

    # BERT hidden dim (auto-detected from model)
    'bert_hidden': 768,                 # will be set after loading BERT

    # Model
    'proj_dim': 256,                    # BERT → projection → LSTM
    'hidden_dim': 512,
    'n_layers': 2,
    'dropout': 0.3,

    # Training
    'batch_size': 128,                  # smaller because BERT is huge
    'epochs': 20,
    'lr': 1e-3,
    'bert_lr': 2e-5,                    # used only when bert_freeze=False
    'clip_grad': 1.0,
    'label_smooth': 0.1,
    'warmup_steps': 300,

    # Decoding
    'beam_width': 4,
    'len_penalty': 0.6,

    # Eval
    'eval_every': 1,
    'bleu_samples': 800,

    # Output
    'ckpt_path': '/kaggle/working/best_bert.pt',
    'out_dir': '/kaggle/working',
}

import glob
search = glob.glob('/kaggle/input/datasets/au23cs060bodhinijain/adivani/**/*.hi', recursive=True)
if search:
    CFG['hi_file'] = os.path.basename(search[0])
    CFG['data_dir'] = os.path.dirname(search[0])
    CFG['mr_file'] = CFG['hi_file'].replace('.hi', '.mr')
    print(f'Auto-detected: {search[0]}')
else:
    print('Update CFG data_dir/hi_file/mr_file manually.')

print('Config ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Load BERT tokenizers & models
# ─────────────────────────────────────────────────────────────────────────────
print('Loading Hindi BERT...')
hi_tokenizer = AutoTokenizer.from_pretrained(CFG['hi_bert'])
hi_bert = AutoModel.from_pretrained(CFG['hi_bert'])

print('Loading Marathi BERT...')
mr_tokenizer = AutoTokenizer.from_pretrained(CFG['mr_bert'])
mr_bert = AutoModel.from_pretrained(CFG['mr_bert'])

# Auto-detect BERT hidden size
CFG['bert_hidden'] = hi_bert.config.hidden_size
print(f'BERT hidden size: {CFG["bert_hidden"]}')

# Move to GPU and optionally freeze
for bert_model, name in [(hi_bert, 'Hindi'), (mr_bert, 'Marathi')]:
    bert_model.to(DEVICE)
    bert_model.eval()  # always eval mode (we use as feature extractor)
    if CFG['bert_freeze']:
        for p in bert_model.parameters():
            p.requires_grad = False
        print(f'{name} BERT: FROZEN')
    else:
        print(f'{name} BERT: TRAINABLE (fine-tune mode)')

print('BERT models loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Load data
# ─────────────────────────────────────────────────────────────────────────────
def load_parallel(hi_path, mr_path, max_samples=None, max_len=100):
    src_lines, tgt_lines = [], []
    with open(hi_path, encoding='utf-8') as f_hi, \
         open(mr_path, encoding='utf-8') as f_mr:
        for hi, mr in zip(f_hi, f_mr):
            hi, mr = hi.strip(), mr.strip()
            if hi and mr and len(hi.split()) <= max_len and len(mr.split()) <= max_len:
                src_lines.append(hi)
                tgt_lines.append(mr)
    if max_samples and len(src_lines) > max_samples:
        idx = random.sample(range(len(src_lines)), max_samples)
        src_lines = [src_lines[i] for i in idx]
        tgt_lines = [tgt_lines[i] for i in idx]
    return src_lines, tgt_lines

hi_path = os.path.join(CFG['data_dir'], CFG['hi_file'])
mr_path = os.path.join(CFG['data_dir'], CFG['mr_file'])
src_lines, tgt_lines = load_parallel(hi_path, mr_path,
                                      CFG['max_samples'], CFG['max_len_words'])
print(f'Loaded {len(src_lines):,} sentence pairs')

n_val = int(len(src_lines) * CFG['val_split'])
n_train = len(src_lines) - n_val
indices = list(range(len(src_lines)))
random.shuffle(indices)
train_idx, val_idx = indices[:n_train], indices[n_train:]

train_src = [src_lines[i] for i in train_idx]
train_tgt = [tgt_lines[i] for i in train_idx]
val_src   = [src_lines[i] for i in val_idx]
val_tgt   = [tgt_lines[i] for i in val_idx]
print(f'Train: {len(train_src):,} | Val: {len(val_src):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Train SentencePiece for TARGET side (Marathi output)
# ─────────────────────────────────────────────────────────────────────────────
# Note: SRC side uses BERT tokenizer; TGT side uses SPM BPE
raw_corpus = '/kaggle/working/spm_bert_input.txt'
with open(raw_corpus, 'w', encoding='utf-8') as f:
    for line in tgt_lines:  # only Marathi for target vocab
        f.write(line + '\n')

spm.SentencePieceTrainer.train(
    input=raw_corpus,
    model_prefix=CFG['spm_model'],
    vocab_size=CFG['vocab_size'],
    character_coverage=0.9995,
    model_type='bpe',
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
)

sp = spm.SentencePieceProcessor()
sp.load(CFG['spm_model'] + '.model')

PAD_ID = sp.pad_id()
BOS_ID = sp.bos_id()
EOS_ID = sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()

print(f'Target vocab size: {VOCAB_SIZE}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Pre-cache BERT embeddings (CRITICAL for speed)
# Caching avoids re-running BERT every epoch — saves 5x time
# ─────────────────────────────────────────────────────────────────────────────
def get_bert_embeddings_batch(sentences, tokenizer, bert_model, device,
                               max_len=64, n_last_layers=4):
    """Returns (embeddings_list, lengths_list) — variable length per sentence."""
    enc = tokenizer(
        sentences,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=max_len,
    )
    input_ids      = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    token_type_ids = enc.get('token_type_ids', None)
    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(device)

    with torch.no_grad(), autocast():
        out = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,
        )
    # Average last n layers
    hidden_states = out.hidden_states[-n_last_layers:]  # list of (B, S, H)
    embeds = torch.stack(hidden_states).mean(dim=0)     # (B, S, H)

    # Return CPU tensors per sentence (excluding [CLS] and [SEP])
    result_embeds, result_lens = [], []
    for i in range(embeds.size(0)):
        mask_i = attention_mask[i].bool()
        ids_i  = input_ids[i][mask_i]  # non-padded token ids
        emb_i  = embeds[i][mask_i]     # (valid_len, H)
        # strip [CLS] and [SEP]
        emb_i = emb_i[1:-1].cpu().float()
        result_embeds.append(emb_i)
        result_lens.append(emb_i.size(0))
    return result_embeds, result_lens


def cache_embeddings(sentences, tokenizer, bert_model, device,
                     max_len=64, n_last_layers=4, batch_size=64, desc=''):
    all_embeds, all_lens = [], []
    for i in tqdm(range(0, len(sentences), batch_size), desc=f'Caching {desc}'):
        batch = sentences[i:i+batch_size]
        embs, lens = get_bert_embeddings_batch(
            batch, tokenizer, bert_model, device, max_len, n_last_layers)
        all_embeds.extend(embs)
        all_lens.extend(lens)
    return all_embeds, all_lens


print('Caching Hindi BERT embeddings (train)...')
train_src_embeds, train_src_lens = cache_embeddings(
    train_src, hi_tokenizer, hi_bert, DEVICE,
    CFG['max_bert_len'], CFG['bert_layers'], batch_size=64, desc='train_hi')

print('Caching Hindi BERT embeddings (val)...')
val_src_embeds, val_src_lens = cache_embeddings(
    val_src, hi_tokenizer, hi_bert, DEVICE,
    CFG['max_bert_len'], CFG['bert_layers'], batch_size=64, desc='val_hi')

print(f'Embeddings cached. Shape sample: {train_src_embeds[0].shape}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Dataset & DataLoader (BERT embeddings as input)
# ─────────────────────────────────────────────────────────────────────────────
class BERTNMTDataset(Dataset):
    def __init__(self, src_embeds, src_lens_list, tgt_lines, sp, max_tgt_len=80):
        self.items = []
        for emb, src_l, tgt in zip(src_embeds, src_lens_list, tgt_lines):
            if src_l == 0:
                continue
            tgt_ids = sp.encode(tgt)
            if 1 <= len(tgt_ids) <= max_tgt_len:
                self.items.append((emb, tgt_ids))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def bert_collate_fn(batch):
    batch.sort(key=lambda x: x[0].size(0), reverse=True)
    src_embs, tgt_seqs = zip(*batch)

    src_lens = torch.tensor([e.size(0) for e in src_embs])
    H = src_embs[0].size(-1)
    max_s = src_lens[0].item()
    src_pad = torch.zeros(len(src_embs), max_s, H)
    for i, emb in enumerate(src_embs):
        src_pad[i, :emb.size(0)] = emb

    tgt_lens = [len(t) + 1 for t in tgt_seqs]
    tgt_in  = torch.zeros(len(tgt_seqs), max(tgt_lens), dtype=torch.long)
    tgt_out = torch.full((len(tgt_seqs), max(tgt_lens)), PAD_ID, dtype=torch.long)
    for i, t in enumerate(tgt_seqs):
        tgt_with_bos = [BOS_ID] + t
        tgt_with_eos = t + [EOS_ID]
        tgt_in[i,  :len(tgt_with_bos)] = torch.tensor(tgt_with_bos)
        tgt_out[i, :len(tgt_with_eos)] = torch.tensor(tgt_with_eos)

    return src_pad, src_lens, tgt_in, tgt_out


train_ds = BERTNMTDataset(train_src_embeds, train_src_lens, train_tgt, sp)
val_ds   = BERTNMTDataset(val_src_embeds,   val_src_lens,   val_tgt,   sp)

effective_batch = CFG['batch_size'] * max(N_GPUS, 1)
train_loader = DataLoader(train_ds, batch_size=effective_batch, shuffle=True,
                          collate_fn=bert_collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=effective_batch, shuffle=False,
                          collate_fn=bert_collate_fn, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Model: BERT-projected LSTM Seq2Seq
# ─────────────────────────────────────────────────────────────────────────────
class BahdanauAttention(nn.Module):
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.W_enc = nn.Linear(enc_hid, dec_hid, bias=False)
        self.W_dec = nn.Linear(dec_hid, dec_hid, bias=False)
        self.v     = nn.Linear(dec_hid, 1, bias=False)

    def forward(self, enc_out, dec_hidden, src_mask):
        energy = torch.tanh(
            self.W_enc(enc_out) + self.W_dec(dec_hidden).unsqueeze(1)
        )
        scores = self.v(energy).squeeze(-1)
        scores = scores.masked_fill(src_mask == 0, -1e4)
        attn_weights = F.softmax(scores, dim=-1)
        context = (attn_weights.unsqueeze(2) * enc_out).sum(1)
        return context, attn_weights


class BERTEncoder(nn.Module):
    """
    Takes pre-computed BERT embeddings (B, S, bert_hidden),
    projects → proj_dim, runs biLSTM.
    """
    def __init__(self, bert_hidden, proj_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.proj    = nn.Sequential(
            nn.Linear(bert_hidden, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.rnn     = nn.LSTM(proj_dim, hidden_dim, n_layers,
                               batch_first=True, dropout=dropout if n_layers > 1 else 0,
                               bidirectional=True)
        self.fc_h    = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_c    = nn.Linear(hidden_dim * 2, hidden_dim)
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim

    def forward(self, src_emb, src_lens):
        projected = self.proj(src_emb)  # (B, S, proj_dim)
        packed    = pack_padded_sequence(projected, src_lens.cpu(),
                                         batch_first=True, enforce_sorted=True)
        out, (h, c) = self.rnn(packed)
        enc_out, _ = pad_packed_sequence(out, batch_first=True)

        h = h.view(self.n_layers, 2, -1, self.hidden_dim)
        c = c.view(self.n_layers, 2, -1, self.hidden_dim)
        h_last = torch.cat([h[-1, 0], h[-1, 1]], dim=-1)
        c_last = torch.cat([c[-1, 0], c[-1, 1]], dim=-1)
        h_init = torch.tanh(self.fc_h(h_last)).unsqueeze(0).repeat(self.n_layers, 1, 1)
        c_init = torch.tanh(self.fc_c(c_last)).unsqueeze(0).repeat(self.n_layers, 1, 1)
        return enc_out, (h_init, c_init)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, enc_hid, dec_hid, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.attention = BahdanauAttention(enc_hid, dec_hid)
        self.rnn       = nn.LSTM(embed_dim + enc_hid, dec_hid, n_layers,
                                  batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc_out    = nn.Linear(dec_hid + enc_hid + embed_dim, vocab_size)
        self.dropout   = nn.Dropout(dropout)

    def forward_step(self, token, hidden, enc_out, src_mask):
        embedded = self.dropout(self.embedding(token.unsqueeze(1)))
        dec_h = hidden[0][-1]
        context, attn_w = self.attention(enc_out, dec_h, src_mask)
        rnn_in = torch.cat([embedded, context.unsqueeze(1)], dim=-1)
        out, hidden = self.rnn(rnn_in, hidden)
        pred = self.fc_out(torch.cat([out.squeeze(1), context, embedded.squeeze(1)], dim=-1))
        return pred, hidden, attn_w

    def forward(self, tgt_in, enc_out, hidden, src_mask):
        B, T = tgt_in.shape
        logits = []
        token = tgt_in[:, 0]
        for t in range(1, T):
            pred, hidden, _ = self.forward_step(token, hidden, enc_out, src_mask)
            logits.append(pred)
            token = tgt_in[:, t]
        return torch.stack(logits, dim=1)


class BERTSeq2Seq(nn.Module):
    def __init__(self, cfg, tgt_vocab_size):
        super().__init__()
        enc_out_dim = cfg['hidden_dim'] * 2  # biLSTM output
        self.encoder = BERTEncoder(
            cfg['bert_hidden'], cfg['proj_dim'],
            cfg['hidden_dim'], cfg['n_layers'], cfg['dropout']
        )
        self.decoder = Decoder(
            tgt_vocab_size, cfg['proj_dim'], enc_out_dim,
            cfg['hidden_dim'], cfg['n_layers'], cfg['dropout']
        )

    def forward(self, src_emb, src_lens, tgt_in):
        enc_out, hidden = self.encoder(src_emb, src_lens)
        # src_mask: 1 where tokens exist (non-padded)
        src_mask = torch.arange(enc_out.size(1), device=enc_out.device).unsqueeze(0) \
                   < src_lens.unsqueeze(1)
        return self.decoder(tgt_in, enc_out, hidden, src_mask)

    def beam_search(self, src_emb, src_lens, beam_width=4, max_len=80, len_penalty=0.6):
        self.eval()
        with torch.no_grad():
            enc_out, hidden = self.encoder(src_emb, src_lens)
            src_mask = torch.arange(enc_out.size(1), device=enc_out.device).unsqueeze(0) \
                       < src_lens.unsqueeze(1)

            beams = [([BOS_ID], hidden, 0.0)]
            completed = []

            for _ in range(max_len):
                new_beams = []
                for tokens, h, score in beams:
                    tok = torch.tensor([tokens[-1]], device=DEVICE)
                    logit, h_new, _ = self.decoder.forward_step(tok, h, enc_out, src_mask)
                    log_probs = F.log_softmax(logit, dim=-1).squeeze(0)
                    top_k = log_probs.topk(beam_width)
                    for lp, idx in zip(top_k.values, top_k.indices):
                        new_seq = tokens + [idx.item()]
                        new_score = score + lp.item()
                        if idx.item() == EOS_ID:
                            norm = ((5 + len(new_seq)) / 6) ** len_penalty
                            completed.append((new_seq, new_score / norm))
                        else:
                            new_beams.append((new_seq, h_new, new_score))

                new_beams.sort(key=lambda x: x[2] / max(len(x[0]), 1), reverse=True)
                beams = new_beams[:beam_width]
                if not beams:
                    break

            if completed:
                completed.sort(key=lambda x: x[1], reverse=True)
                return completed[0][0][1:]
            return beams[0][0][1:] if beams else [EOS_ID]


model = BERTSeq2Seq(CFG, VOCAB_SIZE).to(DEVICE)
if N_GPUS > 1:
    model = nn.DataParallel(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters (excl. BERT): {n_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Loss, optimizer, scheduler
# ─────────────────────────────────────────────────────────────────────────────
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, padding_idx, smoothing=0.1):
        super().__init__()
        self.vocab_size  = vocab_size
        self.padding_idx = padding_idx
        self.smoothing   = smoothing
        self.confidence  = 1.0 - smoothing

    def forward(self, logits, targets):
        log_prob = F.log_softmax(logits, dim=-1)
        smooth_val = self.smoothing / (self.vocab_size - 2)
        with torch.no_grad():
            smooth_dist = torch.full_like(log_prob, smooth_val)
            smooth_dist[:, self.padding_idx] = 0
            smooth_dist.scatter_(1, targets.unsqueeze(1), self.confidence)
            mask = (targets == self.padding_idx)
            smooth_dist[mask] = 0
        loss = (-smooth_dist * log_prob).sum(dim=-1)
        return loss.sum() / (~mask).sum().clamp(min=1)


criterion = LabelSmoothingLoss(VOCAB_SIZE, PAD_ID, CFG['label_smooth'])

# Separate LR groups: BERT (if unfrozen) gets smaller LR
if not CFG['bert_freeze']:
    bert_params = list(hi_bert.parameters()) + list(mr_bert.parameters())
    other_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': bert_params, 'lr': CFG['bert_lr']},
        {'params': other_params, 'lr': CFG['lr']}
    ], betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'],
                                   betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)

def warmup_cosine(step, warmup=300, total=5000):
    if step < warmup:
        return step / max(warmup, 1)
    progress = (step - warmup) / max(total - warmup, 1)
    return max(0.05, 0.5 * (1 + math.cos(math.pi * progress)))

total_steps = CFG['epochs'] * len(train_loader)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda s: warmup_cosine(s, CFG['warmup_steps'], total_steps)
)
scaler = GradScaler()
print('Ready. Optimizer, scheduler, loss function initialized.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Evaluation helpers
# ─────────────────────────────────────────────────────────────────────────────
def eval_loss(model_obj, loader, criterion, device):
    model_obj.eval()
    total_loss, total_tok = 0, 0
    with torch.no_grad():
        for src_emb, src_lens, tgt_in, tgt_out in loader:
            src_emb  = src_emb.to(device)
            src_lens = src_lens.to(device)
            tgt_in   = tgt_in.to(device)
            tgt_out  = tgt_out.to(device)
            with autocast():
                logits = model_obj(src_emb, src_lens, tgt_in)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt_out[:, :T].reshape(B*T))
            n_tok = (tgt_out[:, :T] != PAD_ID).sum().item()
            total_loss += loss.item() * n_tok
            total_tok  += n_tok
    return total_loss / max(total_tok, 1)


def compute_metrics(model_obj, loader, sp, device, n_samples=800, beam_width=4):
    raw = model_obj.module if isinstance(model_obj, nn.DataParallel) else model_obj
    raw.eval()
    hyps, refs = [], []
    count = 0
    with torch.no_grad():
        for src_emb, src_lens, tgt_in, tgt_out in loader:
            src_emb  = src_emb.to(device)
            src_lens = src_lens.to(device)
            B = src_emb.size(0)
            for i in range(B):
                s = src_emb[i:i+1]
                l = src_lens[i:i+1]
                pred_ids = raw.beam_search(s, l, beam_width=beam_width)
                pred_ids = [x for x in pred_ids if x not in (EOS_ID, PAD_ID, BOS_ID)]
                hyps.append(sp.decode(pred_ids))
                ref_ids = [x for x in tgt_out[i].tolist()
                           if x not in (EOS_ID, PAD_ID, BOS_ID)]
                refs.append(sp.decode(ref_ids))
                count += 1
                if count >= n_samples:
                    break
            if count >= n_samples:
                break
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    chrf = sacrebleu.corpus_chrf(hyps, [refs], beta=2).score
    return bleu, chrf, hyps[:5], refs[:5]

print('Eval functions ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Training loop
# ─────────────────────────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'val_loss': [],
    'train_bleu': [], 'val_bleu': [],
    'train_chrf': [], 'val_chrf': [],
}

best_val_bleu = -1

def train_epoch(model_obj, loader, optimizer, criterion, scaler, scheduler, device, clip):
    model_obj.train()
    total_loss, total_tok = 0, 0
    for src_emb, src_lens, tgt_in, tgt_out in tqdm(loader, desc='Train', leave=False):
        src_emb  = src_emb.to(device, non_blocking=True)
        src_lens = src_lens.to(device, non_blocking=True)
        tgt_in   = tgt_in.to(device, non_blocking=True)
        tgt_out  = tgt_out.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model_obj(src_emb, src_lens, tgt_in)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt_out[:, :T].reshape(B*T))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model_obj.parameters(), clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        n_tok = (tgt_out[:, :T] != PAD_ID).sum().item()
        total_loss += loss.item() * n_tok
        total_tok  += n_tok
    return total_loss / max(total_tok, 1)


print('Starting training...')
t0 = time.time()

for epoch in range(1, CFG['epochs'] + 1):
    ep_t0 = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion,
                             scaler, scheduler, DEVICE, CFG['clip_grad'])
    val_loss = eval_loss(model, val_loader, criterion, DEVICE)

    do_eval = (epoch % CFG['eval_every'] == 0)
    if do_eval:
        train_bleu, train_chrf, _, _ = compute_metrics(
            model, train_loader, sp, DEVICE, CFG['bleu_samples'], CFG['beam_width'])
        val_bleu, val_chrf, ex_hyps, ex_refs = compute_metrics(
            model, val_loader, sp, DEVICE, CFG['bleu_samples'], CFG['beam_width'])
    else:
        train_bleu = train_chrf = val_bleu = val_chrf = float('nan')

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_bleu'].append(train_bleu)
    history['val_bleu'].append(val_bleu)
    history['train_chrf'].append(train_chrf)
    history['val_chrf'].append(val_chrf)

    ep_time = time.time() - ep_t0
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:02d}/{CFG["epochs"]} | '
          f'TLoss: {train_loss:.4f} | VLoss: {val_loss:.4f} | '
          f'BLEU: {val_bleu:.2f} | CHRF++: {val_chrf:.2f} | '
          f'LR: {lr_now:.2e} | {ep_time:.0f}s')

    if do_eval and val_bleu > best_val_bleu:
        best_val_bleu = val_bleu
        torch.save({
            'epoch': epoch,
            'model_state': (model.module if N_GPUS > 1 else model).state_dict(),
            'val_bleu': val_bleu,
            'val_chrf': val_chrf,
            'cfg': CFG,
            'history': history,
        }, CFG['ckpt_path'])
        print(f'  ✓ Best BLEU: {val_bleu:.2f} — saved')

    if do_eval:
        print('  Samples:')
        for h, r in zip(ex_hyps[:2], ex_refs[:2]):
            print(f'    HYP: {h}'); print(f'    REF: {r}'); print()

print(f'\nDone. {(time.time()-t0)/60:.1f} min | Best BLEU: {best_val_bleu:.2f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — Plots
# ─────────────────────────────────────────────────────────────────────────────
epochs_x = list(range(1, CFG['epochs'] + 1))
bleu_x = [e for e, v in zip(epochs_x, history['val_bleu']) if not math.isnan(v)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('NMT Training — BERT Embeddings (LSTM + Bahdanau Attention)', fontsize=14, fontweight='bold')

axes[0].plot(epochs_x, history['train_loss'], 'b-o', ms=4, label='Train')
axes[0].plot(epochs_x, history['val_loss'],   'r-o', ms=4, label='Val')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(bleu_x, [v for v in history['train_bleu'] if not math.isnan(v)], 'b-o', ms=4, label='Train')
axes[1].plot(bleu_x, [v for v in history['val_bleu']   if not math.isnan(v)], 'r-o', ms=4, label='Val')
axes[1].set(xlabel='Epoch', ylabel='BLEU-100', title='BLEU-100'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(bleu_x, [v for v in history['train_chrf'] if not math.isnan(v)], 'b-o', ms=4, label='Train')
axes[2].plot(bleu_x, [v for v in history['val_chrf']   if not math.isnan(v)], 'r-o', ms=4, label='Val')
axes[2].set(xlabel='Epoch', ylabel='CHRF++-100', title='CHRF++-100'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CFG['out_dir'], 'bert_emb_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — Final evaluation + save results
# ─────────────────────────────────────────────────────────────────────────────
ckpt = torch.load(CFG['ckpt_path'], map_location=DEVICE)
raw_model = BERTSeq2Seq(CFG, VOCAB_SIZE).to(DEVICE)
raw_model.load_state_dict(ckpt['model_state'])

val_bleu_f, val_chrf_f, hyps_f, refs_f = compute_metrics(
    raw_model, val_loader, sp, DEVICE, n_samples=2000, beam_width=CFG['beam_width'])

print('='*60)
print(f'FINAL RESULTS (BERT Embeddings)')
print(f'  Val BLEU-100  : {val_bleu_f:.2f}')
print(f'  Val CHRF++-100: {val_chrf_f:.2f}')
print(f'  Frozen BERT   : {CFG["bert_freeze"]}')
print(f'  Best epoch    : {ckpt["epoch"]}')
print('='*60)
print('\nQualitative:')
for i, (h, r) in enumerate(zip(hyps_f[:10], refs_f[:10]), 1):
    print(f'  [{i}] HYP: {h}')
    print(f'      REF: {r}')
    print()

results = {
    'model': 'LSTM_BERTEmbeddings',
    'frozen_bert': CFG['bert_freeze'],
    'val_bleu_100': val_bleu_f,
    'val_chrf_100': val_chrf_f,
    'best_epoch': ckpt['epoch'],
    'history': history,
    'qualitative': [{'hyp': h, 'ref': r} for h, r in zip(hyps_f[:20], refs_f[:20])]
}
with open(os.path.join(CFG['out_dir'], 'bert_emb_results.json'), 'w') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('Results saved.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — Inference demo
# ─────────────────────────────────────────────────────────────────────────────
def translate_bert(model_obj, sentence, hi_tokenizer, hi_bert, sp, device, beam_width=4):
    raw = model_obj.module if isinstance(model_obj, nn.DataParallel) else model_obj
    raw.eval()
    embs, lens = get_bert_embeddings_batch(
        [sentence], hi_tokenizer, hi_bert, device,
        CFG['max_bert_len'], CFG['bert_layers'])
    src_emb  = embs[0].unsqueeze(0).to(device)
    src_lens = torch.tensor([lens[0]]).to(device)
    pred_ids = raw.beam_search(src_emb, src_lens, beam_width=beam_width)
    pred_ids = [x for x in pred_ids if x not in (EOS_ID, PAD_ID, BOS_ID)]
    return sp.decode(pred_ids)


demo = [
    'मैं बाज़ार जा रहा हूँ।',
    'आज मौसम बहुत अच्छा है।',
    'वह स्कूल में पढ़ता है।',
    'हमें पानी पीना चाहिए।',
    'यह किताब बहुत रोचक है।'
]

print('Hindi → Marathi (BERT embeddings):')
print('─' * 60)
for s in demo:
    t = translate_bert(raw_model, s, hi_tokenizer, hi_bert, sp, DEVICE)
    print(f'  HI: {s}')
    print(f'  MR: {t}')
    print()